In [ ]:
import os
import subprocess
import sys
import torch

# Ensure the parent Jupyter kernel isn't hoarding VRAM before spawning torchrun
if torch.cuda.is_available() and torch.cuda.memory_allocated(0) > 0:
    raise RuntimeError("Jupyter kernel is holding GPU memory. Click 'Run -> Restart Session' first.")

SRC_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "src" in dirs and os.path.exists(os.path.join(root, "src", "train.py")):
        SRC_ROOT = root
        break

assert SRC_ROOT is not None, "Could not find src/train.py under /kaggle/input"

env = os.environ.copy()
env["PYTHONPATH"] = f"{SRC_ROOT}:{env.get('PYTHONPATH', '')}"
env["TOKENIZERS_PARALLELISM"] = "false"
env["NCCL_P2P_DISABLE"] = "1"
env["NCCL_IB_DISABLE"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

log_path = "/kaggle/working/torchrun.log"
with open(log_path, "w") as log_file:
    res = subprocess.run(
        [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2", "-m", "src.train"],
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

with open(log_path) as f:
    lines = [line.split("\r")[-1] for line in f.readlines() if line.strip()]

print("".join(lines[-60:]))
if res.returncode != 0:
    raise RuntimeError(f"torchrun failed with exit code {res.returncode}")